# DNN Surrogate Capacity Models for RCFs
Reproduction of Xing et al. (2024): *Computer Methods in Applied Mechanics and Engineering*

Trains 8 separate DNNs for outputs: `Dy, Vy, Du, Vu, IDy, IVy, IDu, IVu` 

## 1. Imports

In [ ]:
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

Using device: cpu


## 2. Load Hyperparameters

In [2]:
YAML_PATH = Path('hyperparameters.yaml')

with open(YAML_PATH) as f:
    hp = yaml.safe_load(f)

DS           = hp['dataset']
DNN_DEFAULTS = hp['dnn_default']
OUTS         = hp['dnn_outputs']

TRAIN_SPLIT  = DS['train_split']
R2_THRESHOLD = DS['r2_threshold']
INCREMENT    = DS['increment_per_iter']
MAX_ITER     = DS['max_iterations']

OUTPUT_NAMES = ['Dy', 'Vy', 'Du', 'Vu', 'IDy', 'IVy', 'IDu', 'IVu']

INPUT_COLS = [
    'B', 'D', 'H', 'B_num', 'D_num', 'H_num',
    'colWidth', 'beamDepth', 'beamRat', 'Asc', 'Asb',
    't', 'cover1', 'cover2', 'Ec', 'nu_c', 'fc',
    'fcuRat', 'eps_cu', 'Es', 'nu_s', 'fsy', 'eta'
]

print('Hyperparameters loaded.')
print(f'  total_size={DS["total_size"]:,}  increment={INCREMENT}  max_iter={MAX_ITER}')
print(f'  R2_threshold={R2_THRESHOLD}  train/test={TRAIN_SPLIT}/{1 - TRAIN_SPLIT}')


Hyperparameters loaded.
  total_size=110,649  increment=2000  max_iter=50
  R2_threshold=0.9  train/test=0.8/0.19999999999999996


## 3. Load Data

In [4]:
DATA_PATH = Path('../1. Dataset/DNN_dataset_cleaned.csv')

df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df):,} rows x {len(df.columns)} columns')
df[INPUT_COLS + OUTPUT_NAMES].describe().round(4)

Loaded 110,649 rows x 43 columns


,B,D,H,B_num,D_num,H_num,colWidth,beamDepth,beamRat,Asc,...,fsy,eta,Dy,Vy,Du,Vu,IDy,IVy,IDu,IVu
count,110649.0000,110649.0000,110649.0000,110649.0000,110649.0000,110649.0000,110649.0000,110649.0000,110649.0000,110649.0000,...,110649.0000,110649.0000,110649.0000,110649.0000,110649.0000,110649.0000,110649.0000,110649.0000,110649.0000,110649.0000
mean,6.6375,5.2840,4.1198,3.4788,3.4754,3.4905,0.7893,0.8308,0.4155,423.7656,...,398.2648,0.0237,0.2098,3955.0448,0.6591,5146.9438,0.3132,4063.5094,0.9727,5146.9438
std,1.8185,1.5556,0.6828,0.8678,0.8692,1.5995,0.2478,0.2234,0.0490,213.2455,...,57.6165,0.0092,0.1099,2499.8420,0.4668,3110.7020,0.1539,2575.6090,0.6205,3110.7020
min,3.6000,2.7000,3.0000,2.0000,2.0000,1.0000,0.3000,0.4000,0.3300,60.0170,...,300.0037,0.0080,0.0015,3.5413,0.0247,8.1935,0.0024,3.7656,0.0400,8.1935
25%,5.0633,3.9291,3.5259,2.7247,2.7177,2.1305,0.5865,0.6477,0.3732,239.0700,...,348.3000,0.0157,0.1301,2135.0000,0.3177,2869.6700,0.2000,2194.5900,0.4909,2869.6700
50%,6.5605,5.2215,4.0823,3.4661,3.4607,3.3256,0.8012,0.8415,0.4157,419.5873,...,397.5900,0.0236,0.1864,3416.5000,0.4759,4495.0000,0.2805,3500.5000,0.7387,4495.0000
75%,8.1702,6.6143,4.6903,4.2312,4.2288,4.7044,1.0026,1.0231,0.4580,607.6100,...,447.8131,0.0317,0.2660,5191.9000,0.8839,6736.3000,0.3989,5322.3130,1.3600,6736.3000
max,9.9996,8.1000,5.4000,5.0000,5.0000,6.9992,1.2000,1.2000,0.5000,799.9922,...,500.0000,0.0400,0.9586,23817.8400,2.0003,31377.0000,1.1000,24690.3700,2.3999,31377.0000


## 4. Pre-processing

Per the paper:
- Input features: **StandardScaler** (zero mean, unit variance); fitted on training set only
- Train/test split: uses `split` column from CSV if present, else random 80/20

In [6]:
if 'split' in df.columns:
    train_df = df[df['split'] == 'train'].reset_index(drop=True)
    test_df  = df[df['split'] == 'test'].reset_index(drop=True)
    print(f'Using split column  --  train: {len(train_df):,}  |  test: {len(test_df):,}')
else:
    from sklearn.model_selection import train_test_split
    train_df, test_df = train_test_split(df, test_size=1 - TRAIN_SPLIT, random_state=42)
    train_df = train_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)
    print(f'Random split  --  train: {len(train_df):,}  |  test: {len(test_df):,}')

scaler = StandardScaler()
scaler.fit(train_df[INPUT_COLS])

X_train_all = scaler.transform(train_df[INPUT_COLS]).astype('float32')
X_test      = scaler.transform(test_df[INPUT_COLS]).astype('float32')
Y_train_all = train_df[OUTPUT_NAMES].values.astype('float32')
Y_test      = test_df[OUTPUT_NAMES].values.astype('float32')

print(f'X_train_all: {X_train_all.shape}  |  X_test: {X_test.shape}')

Using split column  --  train: 105,116  |  test: 5,533
X_train_all: (105116, 23)  |  X_test: (5533, 23)


## 5. PyTorch Dataset & DataLoader

In [7]:
class RCFDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def make_loaders(X_tr, y_tr, X_te, y_te, batch_size):
    pin = device.type == 'cuda'
    train_loader = DataLoader(
        RCFDataset(X_tr, y_tr), batch_size=batch_size,
        shuffle=True, pin_memory=pin, num_workers=0
    )
    test_loader = DataLoader(
        RCFDataset(X_te, y_te), batch_size=batch_size * 4,
        shuffle=False, pin_memory=pin, num_workers=0
    )
    return train_loader, test_loader

## 6. DNN Architecture

In [8]:
class DNNModel(nn.Module):
    """
    Fully-connected DNN with BatchNorm + Dropout after each hidden layer.
    neurons: list of hidden-layer widths, e.g. [128, 128, 64, 64, 64, 64]
    Single scalar output per model.
    """
    def __init__(self, n_inputs, neurons, dropout_rate, use_batch_norm):
        super().__init__()
        layers = []
        in_dim = n_inputs
        for width in neurons:
            layers.append(nn.Linear(in_dim, width))
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(width))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            in_dim = width
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def build_model(output_name):
    cfg = OUTS[output_name]
    return DNNModel(
        n_inputs=len(INPUT_COLS),
        neurons=cfg['neurons_per_layer'],
        dropout_rate=cfg['dropout_rate'],
        use_batch_norm=DNN_DEFAULTS['batch_norm'],
    ).to(device)


# Sanity check
_m = build_model('Dy')
print(_m)
del _m


DNNModel(
  (net): Sequential(
    (0): Linear(in_features=23, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=128, out_features=64, bias=True)
    (9): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=64, out_features=64, bias=True)
    (13): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (14): ReLU()
    (15): Dropout(p=0.2, inplace=False)
    (16): Linear(in_features=64, out_features=64, bias=True)
    (17): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_st

## 7. Reusable Training Function

In [9]:
import time

def train_one_model(output_name, X_tr, Y_tr, X_te, Y_te):
    """
    Train a single DNN for one output column.

    Returns
    -------
    model                    : best-checkpoint DNNModel
    train_losses, val_losses : per-epoch MSE lists
    train_r2, test_r2        : final R^2 scores
    y_tr_true, y_te_true     : ground-truth 1-D numpy arrays
    pred_tr, pred_te         : prediction 1-D numpy arrays
    epoch_times              : per-epoch wall-clock times (seconds)
    """
    cfg          = OUTS[output_name]
    batch_size   = cfg['batch_size']
    weight_decay = cfg['weight_decay']
    max_epochs   = DNN_DEFAULTS['max_epochs']
    patience     = DNN_DEFAULTS['early_stopping_patience']
    lr           = DNN_DEFAULTS['initial_lr']

    out_idx = OUTPUT_NAMES.index(output_name)
    y_tr_1d = Y_tr[:, out_idx].reshape(-1, 1)
    y_te_1d = Y_te[:, out_idx].reshape(-1, 1)

    train_loader, test_loader = make_loaders(X_tr, y_tr_1d, X_te, y_te_1d, batch_size)

    model     = build_model(output_name)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    train_losses, val_losses = [], []
    epoch_times = []
    best_val   = float('inf')
    no_improve = 0
    best_state = None

    for epoch in range(max_epochs):
        epoch_start = time.perf_counter()

        # train
        model.train()
        running = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device).squeeze(-1)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running += loss.item() * len(xb)
        train_losses.append(running / len(train_loader.dataset))

        # validate
        model.eval()
        val_running = 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(device)
                yb = yb.to(device).squeeze(-1)
                val_running += criterion(model(xb), yb).item() * len(xb)
        val_loss = val_running / len(test_loader.dataset)
        val_losses.append(val_loss)

        epoch_elapsed = time.perf_counter() - epoch_start
        epoch_times.append(epoch_elapsed)

        # print every 100 epochs
        if (epoch + 1) % 100 == 0:
            with torch.no_grad():
                ep_pred_te = model(torch.from_numpy(X_te).to(device)).cpu().numpy()
            ep_r2 = r2_score(y_te_1d.squeeze(), ep_pred_te)
            print(f'    epoch {epoch+1:4d} | train_loss={train_losses[-1]:.6f} | '
                  f'val_loss={val_loss:.6f} | test R2={ep_r2:.4f} | '
                  f'time={epoch_elapsed:.2f}s')

        # early stopping
        if val_loss < best_val:
            best_val   = val_loss
            no_improve = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'    Early stopping at epoch {epoch+1}.')
                break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        pred_tr = model(torch.from_numpy(X_tr).to(device)).cpu().numpy()
        pred_te = model(torch.from_numpy(X_te).to(device)).cpu().numpy()

    train_r2 = r2_score(y_tr_1d.squeeze(), pred_tr)
    test_r2  = r2_score(y_te_1d.squeeze(), pred_te)

    return (model, train_losses, val_losses,
            train_r2, test_r2,
            y_tr_1d.squeeze(), y_te_1d.squeeze(),
            pred_tr, pred_te,
            epoch_times)


## 8. Iterative Training Loop

For each output, grows the training subset by `increment_per_iter` rows until  
**test R² ≥ threshold** or **max_iterations** is reached.

In [10]:
results = {}

SEP = "=" * 60

for out_name in OUTPUT_NAMES:
    print("\n" + SEP)
    print(f"  Training: {out_name}")
    print(SEP)

    r2_curve     = []
    final_result = None
    best_te_r2   = -float('inf')

    for iteration in range(1, MAX_ITER + 1):
        n_samples = min(iteration * INCREMENT, len(X_train_all))
        X_tr = X_train_all[:n_samples]
        Y_tr = Y_train_all[:n_samples]

        iter_start = time.perf_counter()
        print(f"\n  -- iter {iteration:3d} | n={n_samples:7,} --")

        (model, tr_loss, val_loss,
         tr_r2, te_r2,
         y_tr_true, y_te_true,
         pred_tr, pred_te,
         epoch_times) = train_one_model(out_name, X_tr, Y_tr, X_test, Y_test)

        iter_elapsed = time.perf_counter() - iter_start
        epoch_total  = sum(epoch_times)

        r2_curve.append((n_samples, te_r2))
        print(f"  iter {iteration:3d} DONE | n={n_samples:7,} | "
              f"train R2={tr_r2:.4f} | test R2={te_r2:.4f} | "
              f"iter time={iter_elapsed:.1f}s  (epochs total={epoch_total:.1f}s)")

        if te_r2 > best_te_r2:
            best_te_r2   = te_r2
            final_result = dict(
                model        = model,
                train_losses = tr_loss,
                val_losses   = val_loss,
                train_r2     = tr_r2,
                test_r2      = te_r2,
                y_tr_true    = y_tr_true,
                y_te_true    = y_te_true,
                pred_tr      = pred_tr,
                pred_te      = pred_te,
                r2_curve     = r2_curve,
                final_n      = n_samples,
                epoch_times  = epoch_times,
            )
            print(f"  *** New best model saved (test R2={te_r2:.4f})")
        else:
            print(f"  --- No improvement (best so far: test R2={best_te_r2:.4f})")

        if te_r2 >= R2_THRESHOLD:
            print(f"  >>> R2 threshold {R2_THRESHOLD} reached at n={n_samples:,}. Stopping.")
            break

        if n_samples >= len(X_train_all):
            print("  >>> Full training set used. Stopping.")
            break

    results[out_name] = final_result
    tr2 = final_result["train_r2"]
    te2 = final_result["test_r2"]
    fn  = final_result["final_n"]
    print(f"\n  Best model -- n={fn:,} | train R2={tr2:.4f}  test R2={te2:.4f}")



  Training: Dy
  iter   1 | n=  2,000 | train R2=0.9343 | test R2=0.7198


KeyboardInterrupt: 

## 9. Summary Table

In [ ]:
rows = []
for out_name in OUTPUT_NAMES:
    r = results[out_name]
    rows.append({
        'Output'        : out_name,
        'Final n'       : f"{r['final_n']:,}",
        'Train R2'      : f"{r['train_r2']:.4f}",
        'Test R2'       : f"{r['test_r2']:.4f}",
        'R2>=threshold' : 'YES' if r['test_r2'] >= R2_THRESHOLD else 'NO',
    })
pd.DataFrame(rows).set_index('Output')

## 10. Result Plots

**8 rows** (one per output) × **4 columns**:
1. R² vs dataset size (convergence)
2. Loss vs epoch (train & validation, log scale)
3. Actual vs Predicted: training set
4. Actual vs Predicted: test set

In [ ]:
N_ROWS = len(OUTPUT_NAMES)
N_COLS = 4

fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(22, N_ROWS * 4.5))
fig.suptitle('DNN Surrogate Models -- Training Results',
             fontsize=16, fontweight='bold', y=1.002)

COL_TITLES = [
    'R2 vs Dataset Size',
    'Loss vs Epoch',
    'Pred vs Actual (Train)',
    'Pred vs Actual (Test)',
]
for col, title in enumerate(COL_TITLES):
    axes[0, col].set_title(title, fontsize=11, fontweight='bold', pad=8)

for row_idx, out_name in enumerate(OUTPUT_NAMES):
    r   = results[out_name]
    axs = axes[row_idx]

    axs[0].set_ylabel(out_name, fontsize=12, fontweight='bold', labelpad=6)

    # -- Col 0: R2 convergence --
    sizes, r2s = zip(*r['r2_curve'])
    axs[0].plot(sizes, r2s, 'o-', color='steelblue', lw=1.5, ms=4)
    axs[0].axhline(R2_THRESHOLD, color='red', ls='--', lw=1,
                   label=f'threshold={R2_THRESHOLD}')
    axs[0].set_xlabel('Dataset size')
    axs[0].set_ylabel('Test R2')
    axs[0].set_ylim(max(0, min(r2s) - 0.05), 1.02)
    axs[0].legend(fontsize=8)
    axs[0].grid(True, alpha=0.3)
    axs[0].ticklabel_format(style='sci', axis='x', scilimits=(3, 3))

    # -- Col 1: Loss curve --
    epochs = range(1, len(r['train_losses']) + 1)
    axs[1].plot(epochs, r['train_losses'], label='Train',      color='steelblue',  lw=1.2)
    axs[1].plot(epochs, r['val_losses'],   label='Validation', color='darkorange', lw=1.2)
    axs[1].set_xlabel('Epoch')
    axs[1].set_ylabel('MSE Loss')
    axs[1].set_yscale('log')
    axs[1].legend(fontsize=8)
    axs[1].grid(True, alpha=0.3)

    # -- Cols 2 & 3: Scatter plots --
    scatter_data = [
        (r['y_tr_true'], r['pred_tr'], r['train_r2']),
        (r['y_te_true'], r['pred_te'], r['test_r2']),
    ]
    for col_off, (true_vals, pred_vals, r2_val) in enumerate(scatter_data):
        ax = axs[2 + col_off]
        ax.scatter(true_vals, pred_vals, s=3, alpha=0.2,
                   color='steelblue', rasterized=True)
        lo = min(true_vals.min(), pred_vals.min())
        hi = max(true_vals.max(), pred_vals.max())
        ax.plot([lo, hi], [lo, hi], 'k-', lw=1.2, label='1:1')
        std = (pred_vals - true_vals).std()
        ax.plot([lo, hi], [lo + std, hi + std], 'r--', lw=0.8, label='+/-1 std')
        ax.plot([lo, hi], [lo - std, hi - std], 'r--', lw=0.8)
        ax.set_xlabel(f'True {out_name}')
        ax.set_ylabel(f'Predicted {out_name}')
        ax.text(0.05, 0.92, f'R2 = {r2_val:.4f}',
                transform=ax.transAxes, fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', alpha=0.8))
        ax.legend(fontsize=7, loc='lower right')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig('dnn_training_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: dnn_training_results.png')

## 11. Save Models

In [ ]:
save_dir = Path('saved_models')
save_dir.mkdir(exist_ok=True)

for out_name in OUTPUT_NAMES:
    path = save_dir / f'dnn_{out_name}.pt'
    torch.save(results[out_name]['model'].state_dict(), path)
    print(f'Saved: {path}')
